In [ ]:
from sklearn.tree import DecisionTreeClassifier  # 决策树分类器（作为 AdaBoost 的基学习器）
from sklearn.metrics import accuracy_score

In [ ]:
# 构建弱学习器：决策树桩（max_depth=1，只有一个划分节点）
# AdaBoost 的基学习器通常选择"弱学习器"（略好于随机猜测即可）
# AdaBoost 通过加权组合多个弱学习器来构建强学习器
base_model = DecisionTreeClassifier(max_depth=1, criterion='gini',random_state=1).fit(X_train, y_train)
y_pred = base_model.predict(X_test)
print(f"决策树的准确率：{accuracy_score(y_test,y_pred):.3f}")

In [ ]:
# --- AdaBoost（自适应提升）集成方法 ---
# AdaBoost 核心思想：串行训练多个弱学习器，每个新学习器重点关注前一轮分错的样本
# 算法流程：
# 1. 初始化样本权重均匀分布
# 2. 训练弱学习器 h_t，计算加权错误率 epsilon_t
# 3. 计算弱学习器权重 alpha_t = 0.5 * ln((1-epsilon_t)/epsilon_t)
# 4. 更新样本权重：分错的样本权重增大，分对的减小
# 5. 最终分类器：H(x) = sign(sum(alpha_t * h_t(x)))
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn import metrics
import pandas as pd

wine = load_wine()  # 使用葡萄酒数据集
print(f"所有特征：{wine.feature_names}")
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=1)

print(f"训练数据量：{len(X_train)}，测试数据量：{len(X_test)}")

# 定义 AdaBoost 模型
# n_estimators=50：最多组合50个弱学习器
# learning_rate=0.8：学习率控制每个弱学习器的贡献权重（较小的值需要更多学习器但泛化更好）
model = AdaBoostClassifier(base_estimator=base_model,n_estimators=50,learning_rate=0.8)
model.fit(X_train, y_train)  # 训练
y_pred = model.predict(X_test)  # 预测
acc = metrics.accuracy_score(y_test, y_pred)
print(f"准确率：{acc:.2}")
# AdaBoost 将准确率从0.694（单个弱学习器）提升到0.97

## 使用GridSearchCV自动调参

In [ ]:
# 使用 GridSearchCV 自动搜索最优超参数
# 定义超参数搜索空间：基学习器数量和学习率的组合
hyperparameter_space = {'n_estimators':list(range(2, 102, 2)),  # 基学习器数量：2到100
                        'learning_rate':[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]}  # 学习率

# algorithm='SAMME.R'：使用实数版本的 SAMME 算法，利用预测概率而非硬分类
# SAMME.R 通常比普通 SAMME 收敛更快、性能更好
# cv=5：5折交叉验证，将训练集分为5份，轮流验证
# n_jobs=-1：使用全部 CPU 核心并行计算
gs = GridSearchCV(AdaBoostClassifier(
                                     algorithm='SAMME.R',
                                     random_state=1),
                  param_grid=hyperparameter_space, 
                  scoring="accuracy", n_jobs=-1, cv=5)

gs.fit(X_train, y_train)
print("最优超参数:", gs.best_params_)  # 输出交叉验证选出的最优参数组合